# Import Padstack Definitions
This example shows how to import padstack definitions. This includes

- Download an example board
- Create a config file with hole information
- Create a config file with pad and anti-pad information

## Perform imports and define constants

Perform required imports.

In [1]:
import json
import tempfile
import time
from pathlib import Path

import pandas as pd
import toml
from ansys.aedt.core.examples.downloads import download_file
from IPython.display import display
from pyedb import Edb


Define constants.

In [2]:
AEDT_VERSION = "2026.1"

Download the example PCB data.

In [3]:
temp_folder = tempfile.TemporaryDirectory(suffix=".ansys")
file_edb = download_file(source="edb/ANSYS-HSD_V1.aedb", local_path=temp_folder.name)

## Load example layout.

In [4]:
edbapp = Edb(edbpath=file_edb, version=AEDT_VERSION)

PyEDB INFO: gRPC mode enabled.


PyEDB INFO: Logger is initialized in EDB.


PyEDB INFO: legacy v0.88.dev0


PyEDB INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


WARNING - The installed EDB_RPC_Server does not support shared-memory IPC. Falling back to standard gRPC communication. Update to the latest version of AEDT for improved performance.


PyEDB INFO: Grpc session started: pid=5140


... (3 outputs left out)

PyEDB INFO: Cell main Opened


PyEDB INFO: Refreshing the Components dictionary.


PyEDB INFO: Builder was initialized.


PyEDB INFO: EDB initialized.


## Create a config file with hole information

Keywords

- **name**. Name of the padstack definition.
- **hole_plating_thickness**. Hole plating thickness.
- **hole_range**. Supported types are 'through', 'begin_on_upper_pad', 'end_on_lower_pad', 'upper_pad_to_lower_pad'.
- **hole_parameters**.
  - **shape**. Supported types are 'circle', 'square', 'rectangle'.
  - Other parameters are shape dependent.
    - When shape is 'circle', supported parameter si 'diameter'.
    - When shape is 'square', supported parameter is 'size'.
    - When shape is 'rectangle', supported parameters are 'x_size', 'y_size'.

In [5]:
cfg = dict()
cfg["padstacks"] = {}
cfg["padstacks"]["definitions"] = [
    {
        "name": "v35h15",
        "hole_plating_thickness": "25um",
        "material": "copper",
        "hole_range": "through",
        "hole_parameters": {
            "shape": "circle",
            "diameter": "0.15mm",
        },
    }
]

In [6]:
df = pd.DataFrame(data=cfg["padstacks"]["definitions"])
display(df)

,name,hole_plating_thickness,material,hole_range,hole_parameters
0,v35h15,25um,copper,through,"{'shape': 'circle', 'diameter': '0.15mm'}"


In [7]:
cfg_file_path = Path(temp_folder.name) / "cfg.json"
with open(cfg_file_path, "w") as f:
    json.dump(cfg, f, indent=4, ensure_ascii=False)

Load config file

In [8]:
edbapp.configuration.load(cfg_file_path, apply_file=True)

PyEDB INFO: Updating nets finished. Time lapse 0:00:00.031573


PyEDB INFO: Updating components finished. Time lapse 0:00:00


PyEDB INFO: Creating pin groups finished. Time lapse 0:00:00


PyEDB INFO: Placing sources finished. Time lapse 0:00:00


PyEDB INFO: Applying materials finished. Time lapse 0:00:00


PyEDB INFO: Applying padstack definitions and instances completed in 0.2183 seconds.


... (9 outputs left out)

PyEDB INFO: Caching finished.


PyEDB INFO: Applying setups completed in 0.0000 seconds.


PyEDB INFO: Applying operations completed in 0.0000 seconds.


CfgData(sections=[stackup, nets, padstacks, boundaries, operations])

## Create a config file with pad information

Keywords

- **name**. Name of the padstack definition.
- **pad_parameters**.
  - **regular_pad**. List of pad definition per layer.
    - **layer_name**. Name of the layer.
    - **shape**. Supported types are 'circle', 'square', 'rectangle', 'oval', 'bullet'.
    - Other parameters are shape dependent.
      - When shape is 'circle', supported parameter si 'diameter'.
      - When shape is 'square', supported parameter is 'size'.
      - When shape is 'rectangle', supported parameters are 'x_size', 'y_size'.
      - When shape is 'oval', supported parameters are 'x_size', 'y_size', 'corner_radius'.
      - When shape is 'bullet', supported parameters are 'x_size', 'y_size', 'corner_radius'.

In [9]:
cfg = dict()
cfg["padstacks"] = {}
cfg["padstacks"]["definitions"] = [
    {
        "name": "v35h15",
        "pad_parameters": {
            "regular_pad": [
                {
                    "layer_name": "1_Top",
                    "shape": "circle",
                    "offset_x": "0.1mm",
                    "offset_y": "0.1mm",
                    "rotation": "0",
                    "diameter": "0.5mm",
                },
                {
                    "layer_name": "Inner1(GND1)",
                    "shape": "square",
                    "offset_x": "0.1mm",
                    "offset_y": "0.1mm",
                    "rotation": "45deg",
                    "size": "0.5mm",
                },
            ],
            "anti_pad": [{"layer_name": "1_Top", "shape": "circle", "diameter": "1mm"}],
        },
    }
]

In [10]:
cfg_file_path = Path(temp_folder.name) / "cfg.json"
with open(cfg_file_path, "w") as f:
    json.dump(cfg, f, indent=4, ensure_ascii=False)

Equivalent toml file looks like below

In [11]:
toml_string = toml.dumps(cfg)
print(toml_string)

[padstacks]
[[padstacks.definitions]]
name = "v35h15"

[padstacks.definitions.pad_parameters]
[[padstacks.definitions.pad_parameters.regular_pad]]
layer_name = "1_Top"
shape = "circle"
offset_x = "0.1mm"
offset_y = "0.1mm"
rotation = "0"
diameter = "0.5mm"

[[padstacks.definitions.pad_parameters.regular_pad]]
layer_name = "Inner1(GND1)"
shape = "square"
offset_x = "0.1mm"
offset_y = "0.1mm"
rotation = "45deg"
size = "0.5mm"

[[padstacks.definitions.pad_parameters.anti_pad]]
layer_name = "1_Top"
shape = "circle"
diameter = "1mm"




Load config file

In [12]:
edbapp.configuration.load(cfg_file_path, apply_file=True)

PyEDB INFO: Updating nets finished. Time lapse 0:00:00.031691


PyEDB INFO: Updating components finished. Time lapse 0:00:00


PyEDB INFO: Creating pin groups finished. Time lapse 0:00:00


PyEDB INFO: Placing sources finished. Time lapse 0:00:00


PyEDB INFO: Applying materials finished. Time lapse 0:00:00


PyEDB INFO: Applying padstack definitions and instances completed in 0.2145 seconds.


... (9 outputs left out)

PyEDB INFO: Caching finished.


PyEDB INFO: Applying setups completed in 0.0000 seconds.


PyEDB INFO: Applying operations completed in 0.0000 seconds.


CfgData(sections=[stackup, nets, padstacks, boundaries, operations])

## Save and close Edb
The temporary folder will be deleted once the execution of this script is finished. Replace edbapp.save() with
edbapp.save_as("C:/example.aedb") to keep the example project.

In [13]:
edbapp.save()
edbapp.close()

PyEDB INFO: RPC session released (open databases: 0)


True

In [14]:
# Wait 3 seconds before cleaning the temporary directory.
time.sleep(3)

### Clean up

All project files are saved in the folder ``temp_folder.name``.
If you've run this example as a Jupyter notebook, you
can retrieve those project files. The following cell
removes all temporary files, including the project folder.

In [15]:
temp_folder.cleanup()